# Magpie Client Pipeline Runner — Colab (manual)

Manual, one-client-at-a-time runner for the same logic as the [Streamlit app](https://github.com/diegoakila/client_runner). No scheduling: you run each cell yourself, in order, whenever you want to append a client's data for a given month.

**Flow:** setup & auth → list clients → pick client + month → (optional) edit query → dry-run → confirm & run.

Always run the dry-run cell before the run cell. Nothing is written to BigQuery until you explicitly set `CONFIRM = True` in the last cell.

## 1. Setup & auth

This clones the repo (so we can reuse `pipeline_core.py` instead of duplicating the logic here) and authenticates using your own Google identity via Colab's built-in auth flow — no credentials are stored in this notebook.

In [ ]:
import os

if not os.path.isdir("client_runner"):
    !git clone https://github.com/diegoakila/client_runner.git
%cd client_runner
!git pull -q
!pip install -q google-cloud-bigquery pandas db-dtypes

from google.colab import auth
auth.authenticate_user()

import pipeline_core as core

bq = core.get_client()
print("Connected to project:", core.PROJECT_ID)

## 2. List available clients

Loads the latest generated row per client from `pipeline.client_query`.

In [ ]:
clients_df = core.load_clients(bq)
clients_df[["client", "country", "dataset", "dest_table", "generated_at"]]

## 3. Pick a client and a month

Type a `client` value from the table above and the month you want to append (always the 1st of the month, `YYYY-MM-01`).

In [ ]:
client_name = "leekumkee"  #@param {type:"string"}
month_str = "2026-06-01"  #@param {type:"string"}

match = clients_df[clients_df["client"] == client_name]
assert not match.empty, f"Unknown client '{client_name}'. See the list printed in step 2."
row = match.iloc[0]

dataset = row["dataset"]
dest_table = row["dest_table"]
country = row["country"]
base_query = row["query"]

print("Dataset:    ", dataset)
print("Dest table: ", dest_table)
print("Generated:  ", row["generated_at"])

## 4. (Optional) edit the query

Leave `edited_query` as-is to run the stored version unchanged. Edit the string below only if you need a one-off change — if you do, and later confirm the run in step 6, the edited version is saved as a new row in `pipeline.client_query` so the history stays intact.

In [ ]:
edited_query = base_query  # edit this string if needed

scoped_query = core.build_scoped_query(edited_query, month_str)
print(scoped_query[:1500], "\n..." if len(scoped_query) > 1500 else "")

## 5. Dry-run (always do this before running for real)

Shows the estimated bytes processed, how many rows would be appended, whether rows for this month already exist in the destination table (possible duplicate), and a 50-row sample.

In [ ]:
bytes_processed = core.dry_run(bq, scoped_query)
n_rows = core.row_count(bq, scoped_query)
existing_n = core.existing_month_count(bq, dest_table, month_str)
sample_df = core.preview_rows(bq, scoped_query, limit=50)

print(f"Estimated data processed: {bytes_processed / 1024**2:,.1f} MB")
print(f"Rows that would be appended: {n_rows:,}")
if existing_n > 0:
    print(f"WARNING: {existing_n:,} rows already exist for {month_str} in {dest_table}. "
          "Running will duplicate them (append does not delete existing data).")
elif existing_n == 0:
    print(f"No existing rows for {month_str} in {dest_table} yet.")
else:
    print("Could not check existing rows (destination table may not exist yet).")

sample_df

## 6. Run for real (append into `_dev`)

This only runs if `CONFIRM` is set to `True`. Set it, then re-run this cell. This appends into the client's `_dev` table — it never overwrites or deletes existing data.

In [ ]:
CONFIRM = False  #@param {type:"boolean"}

if not CONFIRM:
    print("Nothing was run. Set CONFIRM = True above and re-run this cell to actually append.")
else:
    if edited_query.strip() != base_query.strip():
        core.save_edited_query(bq, client_name, country, dataset, dest_table, edited_query)
        print("Edited query saved as a new row in pipeline.client_query")

    job = core.append_to_dev(bq, scoped_query, dest_table)
    affected = job.num_dml_affected_rows if job.num_dml_affected_rows is not None else n_rows
    print(f"Appended {affected:,} rows into {dest_table} for month {month_str}.")